# Refine Metadata as part of QC (Part 3)
- Once the sra_growth_metadata_ctherm_curated_p1.tsv is created from Part 2, I manually made changes to it by transferring the data to worksheet: "manual_curation" in the excel file: sra_growth_metadata_ctherm_modified.xlsx. Path: /Users/skd6150/Documents/Research/CBI/iModulon_Ctherm/iModulonMiner/3_quality_control/output_qc/interim
- I changed the eh_chemostat_CI-II and eh_chemostat_CIII-IV to eh_chemostat.
- Now the data needs to be in the format that is required by iModulonMiner.

The next step is to curate the metadata. At a minimum, three new columns must be added to the metadata sheet:
* ``project``: Nickname for the project. Each bioproject should have a unique project IDs.
* ``condition``: Nickname for the experimental condition. Biological/technical replicates must have identical condition IDs.
* ``reference_condition``: Condition ID of the reference condition. Each project has a single reference condition (See [example metadata sheet](https://github.com/SBRG/nf-rnaseq-bacteria/blob/master/example_data/processed_data/metadata_curated.tsv))

Additional columns may include:
* ``strain_description``: The strain name, and any knock-outs or overexpressed genes
* ``base_media``: Media used (e.g. ``M9``)
* ``carbon_source``: Primary carbon source, with concentration in parentheses (e.g. ``glucose(.4%)``). This is usually empty for undefined media.
* ``nitrogen_source``: Primary nitrogen source, with concentration in parentheses (e.g. ``NH4Cl(1M)``). This is usually empty for undefined media.
* ``aerobicity``: Usually ``aerobic`` or ``anaerobic``
* ``treatment``: Any additional supplements or treatments added to the base media (e.g. ``thiamine(0.1M)`` or ``ampicillin(100ug/mL)``)
* ``temperature``
* ``pH``
* ``OD``: Approximate optical density of cells when selected for library preparation
* ``growth_phase``: e.g. ``mid-exponential`` or ``stationary``
* ``culture_type``: Usually ``batch`` or ``chemostat``
* ``skip``: Whether to skip a sample due to external reasons (e.g. not traditional RNA-seq, distant strain, or lack of metadata)

If specific metadata entries are not reported for a sample, these can be left blank. However, if no metadata can be gleaned from public databases, then we recommend discarding the samples.

Once the metadata has been curated, proceed to [Step 3](https://github.com/avsastry/modulome-workflow/edit/main/3_quality_control/expression_QC_part2.ipynb)


## Setup

In [ ]:
import re 

import numpy as np
import pandas as pd

from os import path

### Inputs

get the dataframe using the path of QC2 passed metadata + manual curation

In [ ]:
df_qc2_manual_passed = pd.read_excel('./output_qc/interim/sra_growth_metadata_ctherm_modified.xlsx', sheet_name='manual_curation', index_col=0) #make changes to the sra_growth_metadata_ctherm_modified.xlsx first to create the sheet "manual_curation", if needed to curate
df_qc2_manual_passed.head()

,Run,ReleaseDate,LoadDate,spots,bases,spots_with_mates,avgLength,size_MB,AssemblyName,download_path,...,Unnamed: 100,Unnamed: 101,Unnamed: 102,Unnamed: 103,Unnamed: 104,Unnamed: 105,Unnamed: 106,Unnamed: 107,Unnamed: 108,Unnamed: 109
SRA_Accession,,,,,,,,,,,,,,,,,,,,,
SRX1014220,SRR2002673,2015-11-02 10:20:11,2015-04-29 18:42:39,36341975,1817098750,0,50,726,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SRX1014221,SRR2002674,2015-11-02 10:20:11,2015-04-29 18:42:02,23864175,1193208750,0,50,451,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SRX1014223,SRR2002676,2015-11-02 10:20:11,2015-04-29 18:42:24,30611161,1530558050,0,50,612,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SRX1014225,SRR2002678,2015-11-02 10:20:11,2015-04-29 18:41:57,29139446,1456972300,0,50,590,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SRX1014226,SRR2002679,2015-11-02 10:20:11,2015-04-29 18:42:42,26904972,1345248600,0,50,540,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# create an empty df which will be filled with the data required
cols_for_curated_df = ['project', 'condition', 'reference_condition', 'strain_description', 'base_media', 'carbon_source', 'treatment', 'temperature', 'pH', 'OD', 'growth_phase', 'notes', 'skip']

cols_common_in_df_qc2_manual_passed = [k for k in df_qc2_manual_passed.columns if k in cols_for_curated_df]
cols_not_in_df_curated_p2 = [k for k in cols_for_curated_df if k not in df_qc2_manual_passed.columns]

#common
print(cols_common_in_df_qc2_manual_passed)
# not present
print(cols_not_in_df_curated_p2)

['treatment', 'pH', 'temperature', 'project', 'condition', 'reference_condition', 'skip']
['strain_description', 'base_media', 'carbon_source', 'OD', 'growth_phase', 'notes']


In [10]:
df_curated_p2 = df_qc2_manual_passed.copy()
print(df_curated_p2.shape)

df_curated_p2 = df_curated_p2.loc[df_curated_p2['skip']!=1]
print(df_curated_p2.shape)


for cols in cols_not_in_df_curated_p2:
    df_curated_p2[cols] = ''

df_curated_p2['culture_type'] = ''

(254, 109)
(236, 109)


#### check that the strain names are same and modify the dataset for uniformity

In [ ]:
dict_condition = dict() #SRX to condition

# EH chemostat samples
exp_of_interest = ['SRX3200203', 'SRX3200204', 'SRX3200205', 'SRX3200206', 'SRX3200207', 'SRX3200208', 'SRX3200209', 'SRX5209262', 'SRX5209263', 'SRX5209264', 'SRX5209265', 'SRX5209268', 'SRX5209269', 'SRX5209270', 'SRX5209271', 'SRX5209272', 'SRX5209273', 'SRX5209274', 'SRX5209275', 'SRX5209276', 'SRX5209277', 'SRX5209278', 
                   'SRX5209279', 'SRX5209280', 'SRX5209281', 'SRX5209282', 'SRX5209298', 'SRX5209299', 'SRX5209300', 'SRX5209301', 'SRX7194992', 'SRX7194993', 'SRX7194994', 'SRX7194995', 'SRX7194997', 'SRX7194998', 'SRX7194999', 'SRX7195000', 'SRX7195280', 'SRX7195281', 'SRX7195282', 'SRX7195283', 'SRX7195337', 'SRX7195338', 
                   'SRX7195339', 'SRX7195340', 'SRX7195341', 'SRX7195342', 'SRX7195343', 'SRX7195344', 'SRX7195345', 'SRX7195346', 'SRX7195347', 'SRX7195348', 'SRX7195349', 'SRX7195350', 'SRX7195351', 'SRX7195406', 'SRX7195407', 'SRX7195408', 'SRX7195409', 'SRX7195410', 'SRX7195411', 'SRX7195412', 'SRX7195413', 'SRX7195414', 
                   'SRX7195415', 'SRX7195416', 'SRX7195417', 'SRX7195418', 'SRX7195420', 'SRX7195421', 'SRX7195422', 'SRX7195423', 'SRX7195424', 'SRX7195425', 'SRX7195426', 'SRX7195427', 'SRX7195428', 'SRX7195429', 'SRX7195430', 'SRX7195431', 'SRX7195432', 'SRX7195433', 'SRX7195434', 'SRX7195435', 'SRX7195436', 'SRX7195437', 
                   'SRX7195438', 'SRX7195439', 'SRX7195440', 'SRX7195442', 'SRX7195443', 'SRX7195444', 'SRX7195445', 'SRX7195447', 'SRX7195448', 'SRX7195449', 'SRX7195450', 'SRX7195451', 'SRX7195452', 'SRX7195453', 'SRX7195454', 'SRX7195455', 'SRX7195456', 'SRX7195457', 'SRX7195458', 'SRX7195459', 'SRX7195460', 'SRX7195461', 'SRX7195462']
for idx in exp_of_interest:
    strain_automated = df_curated_p2.loc[idx, 'genotype']
    strain_manual_curated = df_curated_p2.loc[idx, 'condition']
    treatment = df_curated_p2.loc[idx, 'treatment']
    
    if isinstance(treatment, float):
        treatment = 'clim'
    else:
        treatment = 'nlim'

    strain_automated = strain_automated.replace('delta-', 'del_')
    strain_automated = strain_automated.replace('::', '_ins_')
    strain_automated = strain_automated.replace('; ', '_')
    strain_automated = strain_automated.replace('evolved pH auxostat ', 'evo_ph_aux_')
    strain_automated = strain_automated.replace('evolved serial ', 'evo_serial_')
    strain_automated = strain_automated.replace('evolved chemostat ', 'evo_chemostat_')
    
    strain_automated = strain_automated.replace('ldh-S161R', 'ldh_mut-S161R')
    strain_automated = strain_automated.replace('adhE-D494G', 'adhE_mut-D494G')
    
    strain_automated = strain_automated + f'_{treatment}'
    
    
    dict_condition[idx] = strain_automated
    
    if strain_automated.lower()!=strain_manual_curated.lower():
        print(idx, strain_automated, strain_manual_curated)


# lacI
exp_of_interest = ['SRX1014220', 'SRX1014221', 'SRX1014223', 'SRX1014225', 'SRX1014226', 'SRX1014227', 'SRX1014228', 'SRX1014230', 'SRX1014231', 'SRX1014232', 'SRX1014233', 'SRX1014234', 'SRX1014235', 'SRX1014236', 'SRX1014237', 'SRX1014238', 'SRX1014239', 'SRX1014240', 'SRX1014241', 'SRX1014244', 'SRX1014245', 'SRX1014247', 'SRX1014248', 'SRX1014250', 'SRX1014252', 'SRX1014253']

for idx in exp_of_interest:
    strain_automated = df_curated_p2.loc[idx, 'genotype']
    strain_manual_curated = df_curated_p2.loc[idx, 'condition']
    treatment = df_curated_p2.loc[idx, 'time point sampled']
    if treatment in ['mid-log', 'late-log']:
        treatment = treatment.replace('-', '')
    elif treatment=='stationary':
        treatment = 'sta'
    
    strain_automated = strain_automated.replace('delta-', 'del_')
    strain_automated = strain_automated.replace('; ', '_')
    strain_automated = strain_automated + f'_{treatment}'
    
    dict_condition[idx] = strain_automated

    if strain_automated.lower()!=strain_manual_curated.lower():
        print(idx, strain_automated, strain_manual_curated)


# xyl_evo
exp_of_interest = ['SRX1427049', 'SRX1427050', 'SRX1427051', 'SRX1427052', 'SRX1427053', 'SRX1427055']
for idx in exp_of_interest:
    strain_automated = df_curated_p2.loc[idx, 'genotype']
    strain_manual_curated = df_curated_p2.loc[idx, 'condition']
    treatment = df_curated_p2.loc[idx, 'csource']
    
    if treatment == '0 g/L xylose':
        treatment = 'xyl0'
    elif treatment=='15 g/L xylose':
        treatment = 'xyl15'


    strain_automated = strain_automated.replace('delta-', 'del_')
    strain_automated = strain_automated.replace('::', '_ins_')
    strain_automated = strain_automated.replace('; ', '_')
    strain_automated = strain_automated.replace('evolved ', 'evo_')
    
    strain_automated = strain_automated + f'_{treatment}'
    
    dict_condition[idx] = strain_automated

    if strain_automated.lower()!=strain_manual_curated.lower():
        print(idx, strain_automated, strain_manual_curated)

# del_gluS
exp_of_interest = ['SRX1597719', 'SRX1597721', 'SRX1597722', 'SRX1597724']
for idx in exp_of_interest:
    strain_automated = df_curated_p2.loc[idx, 'genotype']
    strain_manual_curated = df_curated_p2.loc[idx, 'condition']
    treatment = df_curated_p2.loc[idx, 'csource']
    
    if treatment == '0 g/L xylose':
        treatment = 'xyl0'
    elif treatment=='15 g/L xylose':
        treatment = 'xyl15'


    strain_automated = strain_automated.replace('delta-', 'del_')
    strain_automated = strain_automated.replace('::', '_ins_')
    strain_automated = strain_automated.replace('; ', '_')
    strain_automated = strain_automated.replace('evolved ', 'evo_')
    
    # strain_automated = strain_automated + f'_{treatment}'
    
    dict_condition[idx] = strain_automated

    if strain_automated.lower()!=strain_manual_curated.lower():
        print(idx, strain_automated, strain_manual_curated)


# evo_ag553_ag601
exp_of_interest = ['SRX1599894', 'SRX1599895', 'SRX1599896', 'SRX1599897', 'SRX1599898', 'SRX1599899', 'SRX1599901', 'SRX1599903', 'SRX1599904', 'SRX1599907', 'SRX1599909', 'SRX1599912']
for idx in exp_of_interest:
    strain_automated = df_curated_p2.loc[idx, 'genotype']
    strain_manual_curated = df_curated_p2.loc[idx, 'condition']
    treatment = df_curated_p2.loc[idx, 'treatment']
    

    if treatment == 'No formate':
        treatment = 'for0'
    elif treatment=='2 mM Formate':
        treatment = 'for2'


    strain_automated = strain_automated.replace('delta-', 'del_')
    strain_automated = strain_automated.replace('::', '_ins_')
    strain_automated = strain_automated.replace('; ', '_')
    strain_automated = strain_automated.replace('evolved ', 'evo_')
    
    strain_automated = strain_automated + f'_{treatment}'
    
    dict_condition[idx] = strain_automated

    if strain_automated.lower()!=strain_manual_curated.lower():    
        print(idx, strain_automated, strain_manual_curated)

# pentose_sugars
exp_of_interest = ['SRX25920392', 'SRX25920393', 'SRX25920394', 'SRX25920395', 'SRX25920396', 'SRX25920397', 'SRX25920398', 'SRX25920399', 'SRX25920400', 'SRX25920401', 'SRX25920402', 'SRX25920403', 'SRX25920404', 'SRX25920405', 'SRX25920406', 'SRX6853642', 'SRX6853643', 'SRX6853644', 'SRX6853645', 'SRX6853646', 'SRX6853647']
for idx in exp_of_interest:
    strain_automated = df_curated_p2.loc[idx, 'genotype']
    strain_manual_curated = df_curated_p2.loc[idx, 'condition']
    treatment = df_curated_p2.loc[idx, 'csource']

    if treatment.lower()=='cellobiose':
        treatment = 'cb'
    elif treatment=='xylose':
        treatment = 'xyl'
    elif treatment=='xylose plus xylan':
        treatment = 'xyl_xyla'
        
    strain_automated = strain_automated.replace('delta-', 'del_')
    strain_automated = strain_automated.replace('::', '_ins_')
    strain_automated = strain_automated.replace('; ', '_')
    strain_automated = strain_automated.replace('evolved (xylose) ', 'evo_xyl_')
    strain_automated = strain_automated.replace('evolved (xylose+xylan) ', 'evo_xyl_xyla_')
    
    strain_automated = strain_automated + f'_{treatment}'
    
    dict_condition[idx] = strain_automated

    if strain_automated.lower()!=strain_manual_curated.lower():    
        print(idx, strain_automated, strain_manual_curated)

# sulfate
exp_of_interest = ['SRX2919058', 'SRX2919060', 'SRX2919061', 'SRX2919062', 'SRX2919063', 'SRX2919064', 'SRX2919065', 'SRX2919066', 'SRX2919067', 'SRX2919068', 'SRX2919069']
for idx in exp_of_interest:
    strain_automated = df_curated_p2.loc[idx, 'genotype']
    strain_manual_curated = df_curated_p2.loc[idx, 'condition']
    treatment = df_curated_p2.loc[idx, 'sulfur source']

    if treatment.lower()=='cysteine':
        treatment = 'cys'
    elif treatment=='sulfate':
        treatment = 'sulf'
        
    strain_automated = strain_automated.replace('delta-', 'del_')
    strain_automated = strain_automated.replace('::', '_ins_')
    strain_automated = strain_automated.replace('; ', '_')
    strain_automated = strain_automated.replace('evolved (xylose) ', 'evo_xyl_')
    strain_automated = strain_automated.replace('evolved (xylose+xylan) ', 'evo_xyl_xyla_')
    
    strain_automated = strain_automated + f'_{treatment}'
    
    dict_condition[idx] = strain_automated

    if strain_automated.lower()!=strain_manual_curated.lower():    
        print(idx, strain_automated, strain_manual_curated)

# ll1210
exp_of_interest = ['SRX3042243', 'SRX3042244', 'SRX3042245', 'SRX3042246', 'SRX3042247', 'SRX3042248', 'SRX3042249', 'SRX3042250', 'SRX3042251', 'SRX3042252', 'SRX3042253', 'SRX3042254']
for idx in exp_of_interest:
    strain_automated = df_curated_p2.loc[idx, 'genotype']
    strain_manual_curated = df_curated_p2.loc[idx, 'condition']
    treatment = str(df_curated_p2.loc[idx, 'pH'])
    treatment = re.sub(r'6\.(\d+)', r'ph6\1', treatment)

        
    strain_automated = strain_automated.replace('delta-', 'del_')
    strain_automated = strain_automated.replace('::', '_ins_')
    strain_automated = strain_automated.replace('; ', '_')
    strain_automated = strain_automated.replace('evolved (xylose) ', 'evo_xyl_')
    strain_automated = strain_automated.replace('evolved (xylose+xylan) ', 'evo_xyl_xyla_')
    
    strain_automated = strain_automated + f'_{treatment}'
    
    dict_condition[idx] = strain_automated

    if strain_automated.lower()!=strain_manual_curated.lower():    
        print(idx, strain_automated, strain_manual_curated)


# cell_stressor
exp_of_interest = ['SRX5085503', 'SRX5085504', 'SRX5085505', 'SRX5085506', 'SRX5085507', 'SRX5085508', 'SRX5085509', 'SRX5085511', 'SRX5085512', 'SRX5085513', 'SRX5085514', 'SRX5085517', 'SRX5085518', 'SRX5085519', 'SRX5085520', 'SRX5085521', 'SRX5085522', 'SRX5085523', 'SRX5085524', 'SRX5085525', 'SRX5085526', 'SRX5085527', 'SRX5085528', 'SRX5085529', 'SRX5085530', 'SRX5085531']
for idx in exp_of_interest:
    strain_automated = df_curated_p2.loc[idx, 'temperature']
    strain_manual_curated = df_curated_p2.loc[idx, 'condition']

    media = str(df_curated_p2.loc[idx, 'growth_medium']).lower()
    treatment = df_curated_p2.loc[idx, 'csource']

    
    if isinstance(treatment, float):
        treatment = ''
    elif treatment=='cellulose':
        treatment = 'cel'

    strain_automated = strain_automated.replace('oC', 'T')
    strain_automated = strain_automated + f'_{media}'
    strain_automated = strain_automated + f'_{treatment}'
    strain_automated = strain_automated.rstrip('_')

    dict_condition[idx] = strain_automated
    
    if strain_automated.lower()!=strain_manual_curated.lower():
        print(idx, strain_automated, strain_manual_curated)

SRX7194998 del_hpt_del_spo0A_clim del_hpt_ldh_mut-S161R_del_adhE_ins_adhE_mut-D494G
SRX1014240 del_hpt_del_Clo1313_2023_latelog del_hpt_del_Clo1313_2023_midlog


In [13]:
for idx in df_curated_p2.index.to_list():

    try:
        df_curated_p2.loc[idx, 'strain_description'] = df_curated_p2.loc[idx, 'organism'].strip('') + ' ' + df_curated_p2.loc[idx, 'genotype'].strip('')
    except (TypeError, AttributeError):
        df_curated_p2.loc[idx, 'strain_description'] = df_curated_p2.loc[idx, 'organism'].strip('')
    
    try:
        df_curated_p2.loc[idx, 'condtion'] = dict_condition[idx]
    except KeyError:
        print(f'{idx} not in dictionary')
    
    try:
        df_curated_p2.loc[idx, 'base_media'] = df_curated_p2.loc[idx, 'growth_medium'].strip('')
    except AttributeError:
        df_curated_p2.loc[idx, 'base_media'] = ''
    
    try:
        df_curated_p2.loc[idx, 'carbon_source'] = df_curated_p2.loc[idx, 'csource'].strip('')
    except AttributeError:
        df_curated_p2.loc[idx, 'carbon_source'] = ''

    if int(df_curated_p2.loc[idx, 'batch'])==1:
        df_curated_p2.loc[idx, 'culture_type'] = 'batch'
    else:
        df_curated_p2.loc[idx, 'culture_type'] = 'chemostat'

    
    if int(df_curated_p2.loc[idx, 'batch'])==1:
        try:
            df_curated_p2.loc[idx, 'growth_phase'] = df_curated_p2.loc[idx, 'time point sampled'].strip('')
        except AttributeError:
            df_curated_p2.loc[idx, 'growth_phase'] = ''
    else:
        if isinstance(df_curated_p2.loc[idx, 'time point sampled'], str):
            print(idx, df_curated_p2.loc[idx, 'time point sampled'])
        
        df_curated_p2.loc[idx, 'growth_phase'] = ''

    try:
        df_curated_p2.loc[idx, 'notes'] = df_curated_p2.loc[idx, 'comments'].strip('')
    except AttributeError:
        df_curated_p2.loc[idx, 'notes'] = ''

    if np.isnan(df_curated_p2.loc[idx, 'skip']):
        df_curated_p2.loc[idx, 'skip'] = 'FALSE'
    else:
        df_curated_p2.loc[idx, 'skip'] = 'TRUE'

SRX2193597 not in dictionary
SRX2193598 not in dictionary
SRX2193599 not in dictionary
SRX2193600 not in dictionary
SRX2193601 not in dictionary
SRX2193603 not in dictionary
SRX2193604 not in dictionary
SRX3042243 0.1h-1
SRX3042244 0.1h-1
SRX3042245 0.1h-1
SRX3042246 0.1h-1
SRX3042247 0.1h-1
SRX3042248 0.1h-1
SRX3042249 0.1h-1
SRX3042250 0.1h-1
SRX3042251 0.1h-1
SRX3042252 0.1h-1
SRX3042253 0.1h-1
SRX3042254 0.1h-1
SRX5209262 0.1h-1
SRX5209263 0.1h-1
SRX5209264 0.1h-1
SRX5209265 0.1h-1
SRX5209268 0.1h-1
SRX5209269 0.1h-1
SRX5209270 0.1h-1
SRX5209271 0.1h-1
SRX5209272 0.1h-1
SRX5209273 0.1h-1
SRX5209274 0.1h-1
SRX5209275 0.1h-1
SRX5209276 0.1h-1
SRX5209277 0.1h-1
SRX5209278 0.1h-1
SRX5209279 0.1h-1
SRX5209280 0.1h-1
SRX5209281 0.1h-1
SRX5209282 0.1h-1
SRX5209298 0.1h-1
SRX5209299 0.1h-1
SRX5209300 0.1h-1
SRX5209301 0.1h-1
SRX7194992 0.1h-1
SRX7194993 0.1h-1
SRX7194994 0.1h-1
SRX7194995 0.1h-1
SRX7194997 0.1h-1
SRX7194998 0.1h-1
SRX7194999 0.1h-1
SRX7195000 0.1h-1
SRX7195280 0.1h-1
SRX71

/var/folders/8c/yy8ts7rn59j58vl5_l6jgcmd1lvs3b/T/ipykernel_44870/3353082636.py:46: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'FALSE' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_curated_p2.loc[idx, 'skip'] = 'FALSE'


In [14]:
# remove columns that are in essence duplicates of the new columns
remove_cols = ['organism', 'genotype', 'growth_medium', 'csource', 'batch', 'time point sampled', 'comments']

df_curated_p2.drop(remove_cols, axis=1, inplace=True)

In [15]:
df_curated_p2.to_csv('./output_qc/interim/sra_growth_metadata_ctherm_curated_p2.tsv', sep='\t') #first curation with curated file + SRA curation + manual curation

In [16]:
len(df_curated_p2.condition.unique())

59

### FIN